## #6 Cell Type Dynamics

## Purpose

Get number, size, and timing of cell type fate restricted clades

## Setup

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import seaborn as sns
from devmap.config import get_paths, lineage_palette, set_theme
from devmap.fate import annotate_commitment, extant_transition_count
from devmap.progenitors import identify_fate_progenitors
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("fate")

## Load data

In [2]:
tdata = load_data("topology")
cell_types = pd.read_csv(base_path / "data" / "cell_types.csv", index_col=0)
cell_type_info = cell_types.drop_duplicates("cell_type").set_index("cell_type")
order = list(reversed(pd.read_csv(results_path / "e95_linkage_order.csv", index_col=None)["target"].to_list()))

## Identify fate-restricted clades

In [7]:
one_hot = pd.get_dummies(tdata.obs["cell_type"]).astype(int)
tdata.obsm["cell_type_counts"] = one_hot
tdata.obs["n"] = 1
py.tl.ancestral_states(tdata, keys = "cell_type_counts", method = "sum")
py.tl.ancestral_states(tdata, keys = "n", method = "sum")

In [ ]:
clone_progenitors = {}
leaf_assignments = []
for clone, tree in tdata.obst.items():
    print(f"Processing clone {clone}...")
    progenitors, clone_assignements = identify_fate_progenitors(
        tree,
        "cell_type_counts",
        fate_names = one_hot.columns)
    clone_progenitors[clone] = progenitors.assign(clone=clone)
    leaf_assignments.append(clone_assignements)
progenitors = pd.concat(clone_progenitors.values())
leaf_assignments = pd.concat(leaf_assignments)
progenitors["stage"] = progenitors["clone"].str.split("-R").str[0]
progenitors["embryo"] = progenitors["clone"].str.split("-C").str[0]
progenitors["lineage"] = progenitors["fate"].map(cell_type_info["lineage"])

## Clade stats

In [16]:
prog_stats = (
    progenitors
    .query("total_descendants > 1 & stage == 'E9.5'")
    .groupby(["embryo", "stage", "fate", "lineage"])
    .agg(
        mean_time=("time", "mean"),
        mean_size=("fate_descendants", "mean"),
        n_progenitors=("node", "nunique"),
    )
    .reset_index()
)

Number of clades

In [ ]:
fig, ax = plt.subplots(figsize=(.8, 5.85), dpi=600, sharex=True)
df = prog_stats.query("stage == 'E9.5' & fate in @order")
# sort by order
df["fate"] = pd.Categorical(df["fate"], categories=order, ordered=True)
sns.barplot(data = df, y = "fate", x = "n_progenitors", hue = "lineage",legend = False,
                palette=lineage_palette, saturation = 1, ax=ax,edgecolor="black", dodge=False,
                linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"})
plt.yticks(fontsize=5);
plt.ylabel("")
sns.scatterplot(data = df, y = "fate", x = "n_progenitors", color = "black", s = 3,
                legend = False, ax=ax, edgecolor="none")
plt.ylim(-.5,len(order) - .5)
#plt.xscale("log")
save_plot(plots_path / "e95_n_progenitors_by_cell_type.svg", fig, rasterize=False)

Clade size

In [ ]:
fig, ax = plt.subplots(figsize=(.8, 5.85), dpi=600, sharex=True)
df = prog_stats.query("stage == 'E9.5' & fate in @order")
df["fate"] = pd.Categorical(df["fate"], categories=order, ordered=True)
sns.barplot(data = df, y = "fate", x = "mean_size", hue = "lineage",legend = False,
                palette=lineage_palette, saturation = 1, ax=ax,edgecolor="black", dodge=False,
                linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"}) 
plt.yticks(fontsize=5);
plt.ylabel("")
sns.scatterplot(data = df, y = "fate", x = "mean_size", color = "black", s = 3,
                legend = False, ax=ax, edgecolor="none")
plt.ylim(-.5,len(order) - .5)
save_plot(plots_path / "e95_prog_size_by_cell_type.svg", fig, rasterize=False)

Time

In [ ]:
fig, ax = plt.subplots(figsize=(.8, 5.85), dpi=600, sharex=True)
df = prog_stats.query("stage == 'E9.5' & fate in @order")
df["fate"] = pd.Categorical(df["fate"], categories=order, ordered=True)
sns.boxplot(data = df, y = "fate", x = "mean_time", hue = "lineage",legend = False,
                palette=lineage_palette, saturation = 1, ax=ax,dodge = False,
                linewidth=0.5, linecolor="black") 
sns.scatterplot(data = df, y = "fate", x = "mean_time", color = "black", s = 3, 
                legend = False, ax=ax, edgecolor="none", zorder = 10)
ax.set_xlabel("Mean time")
plt.yticks(fontsize=5);
plt.ylim(-.5,len(order) - .5)
plt.ylabel("")
save_plot(plots_path / "e95_prog_time_by_cell_type.svg", fig, rasterize=False)

Save results

In [ ]:
progenitors.to_csv(results_path / "cell_type_progenitors.csv")
leaf_assignments.to_csv(results_path / "cell_type_progenitor_descendants.csv")
prog_stats.to_csv(results_path / "e95_cell_type_progenitor_stats.csv")

## Cell subtypes clade stats

In [38]:
one_hot = pd.get_dummies(tdata.obs["cell_subtype"]).astype(int)
tdata.obsm["cell_subtype_counts"] = one_hot
tdata.obs["n"] = 1
py.tl.ancestral_states(tdata, keys = "cell_subtype_counts", method = "sum")
py.tl.ancestral_states(tdata, keys = "n", method = "sum")

Get clades

In [ ]:
clone_progenitors = {}
leaf_assignments = []
for clone, tree in tdata.obst.items():
    print(f"Processing clone {clone}...")
    progenitors, clone_assignements = identify_fate_progenitors(
        tree,
        "cell_subtype_counts",
        fate_names = one_hot.columns)
    clone_progenitors[clone] = progenitors.assign(clone=clone)
    leaf_assignments.append(clone_assignements)
progenitors = pd.concat(clone_progenitors.values())
leaf_assignments = pd.concat(leaf_assignments)
progenitors["stage"] = progenitors["clone"].str.split("-R").str[0]
progenitors["embryo"] = progenitors["clone"].str.split("-C").str[0]
progenitors["lineage"] = progenitors["fate"].map(cell_type_info["lineage"])
prog_stats = (
    progenitors
    .query("total_descendants > 1 & stage == 'E9.5'")
    .groupby(["embryo", "stage", "fate", "lineage"])
    .agg(
        mean_time=("time", "mean"),
        mean_size=("fate_descendants", "mean"),
        n_progenitors=("node", "nunique"),
    )
    .reset_index()
)

Save results

In [43]:
progenitors.to_csv(results_path / "cell_subtype_progenitors.csv")
leaf_assignments.to_csv(results_path / "cell_subtype_progenitor_descendants.csv")
prog_stats.to_csv(results_path / "e95_cell_subtype_progenitor_stats.csv")

## Fate restriction over time

In [3]:
stage_tdata = tdata[tdata.obs.stage == "E9.5"].copy()
germ_assignment = stage_tdata.obs["germ_layer"].replace({"Epiblast": "Other"}).to_dict()
lineage_assignment = stage_tdata.obs["lineage"].replace({"Epiblast": "Other", "Mesoderm": "Mesoderm (other)"}).to_dict()
type_assignment = stage_tdata.obs["cell_type"].to_dict()

Lineage level

In [ ]:
for tree in stage_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment, lineage_assignment], threshold=0.9, exclude_categories=["Other"])
paths = extant_transition_count(stage_tdata, key = "commitment", depth_key="time", bins = np.arange(0, 10, .5))
paths["count"] = np.ceil(paths["count"] / 3).astype(int)
paths.to_csv(results_path / "e95_lineage_fate_restriction.csv", index=False)

Cell type level

In [ ]:
for tree in stage_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment, lineage_assignment, type_assignment], threshold=0.9, exclude_categories=["Other"])
paths = extant_transition_count(stage_tdata, key = "commitment", depth_key="time", bins = np.arange(0, 10, .5))
paths["count"] = np.ceil(paths["count"] / 3).astype(int)
paths.to_csv(results_path / "e95_type_fate_restriction.csv", index=False)